# Explore the COAWST US East Coast and Gulf of Mexico Forecast Archive Dataset
This is a cloud-optimized version of the NetCDF files accessed from the USGS ScienceBase item [Collection of COAWST model forecast for the US East Coast and Gulf of Mexico](https://www.sciencebase.gov/catalog/item/610acd4fd34ef8d7056893da).   The original daily forecast files were converted into weekly NetCDF files with 168 points in the time dimension to facilitate time series access.

This notebook accesses the data via an [Icechunk](https://icechunk.io) virtual store — a cloud-native, version-controlled array store built on top of the original S3 NetCDF files.

In [ ]:
import numpy as np
import xarray as xr
import zarr
import icechunk
from icechunk import (
    ManifestConfig,
    ManifestSplitCondition,
    ManifestSplitDimCondition,
    ManifestSplittingConfig,
)
import hvplot.xarray
import cf_xarray
import panel as pn
from matplotlib import path
import xoak

## Open Dataset

Open the Icechunk store anonymously (no credentials required for read access). Metadata and coordinate data are loaded, but not the actual data variables — those are loaded only as needed.

In [ ]:
bucket = 'usgs-coawst'
region = 'us-west-2'
prefix = 'useast-archive/icechunk/coawst-useast.icechunk'
TIME_DIM = 'ocean_time'

split_config = ManifestSplittingConfig.from_dict({
    ManifestSplitCondition.AnyArray(): {
        ManifestSplitDimCondition.DimensionName(TIME_DIM): 365 * 24
    }
})
config = icechunk.RepositoryConfig(manifest=ManifestConfig(splitting=split_config))
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f's3://{bucket}/',
        store=icechunk.s3_store(region=region, anonymous=True),
    )
)
creds = icechunk.containers_credentials(
    {f's3://{bucket}/': icechunk.s3_credentials(anonymous=True)}
)
storage = icechunk.s3_storage(bucket=bucket, prefix=prefix, region=region, anonymous=True)
repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=creds)
session = repo.readonly_session('main')

In [ ]:
%%time
ds = xr.open_zarr(session.store, consolidated=False)
ds

In [ ]:
ds.nbytes / 1e12

In [ ]:
ds['ocean_time'].attrs

We can explore a specific variable of interest:

In [ ]:
var = 'Hwave'
da = ds[var]
da

Use CF conventions to identify longitude, latitude, and time coordinate variables:

In [ ]:
x = da.cf['longitude']
y = da.cf['latitude']
t = da.cf['time']
print(x.name, y.name, t.name)

## Example: Load the entire spatial domain for a variable at a specific time step
Loading the entire spatial domain at a time step only requires reading 8 chunks of data, so it loads in a few seconds. A Dask cluster doesn't help much here as it's already fast.

In [ ]:
%%time
da2d = da.cf.sel(T='2012-10-29 12:00', method='nearest').load()

In [ ]:
da2d.hvplot.quadmesh(x=x.name, y=y.name, rasterize=True, geo=True, tiles='OSM', cmap='viridis')

## Example: Load a time series for a variable at a specific lon,lat location for a specified time range

To identify a point by lat/lon we use the `xoak` package, since lon/lat are 2D coordinate arrays:

In [ ]:
lat, lon = 42.5, -70.0  # Gulf of Maine, 100km east of Boston, MA

In [ ]:
da.xoak.set_index([y.name, x.name], 'scipy_kdtree')

In [ ]:
ds_point = xr.Dataset({'lon': ('point', [lon]), 'lat': ('point', [lat])})

In [ ]:
%%time
da1d = da.xoak.sel(lat_rho=ds_point.lat, lon_rho=ds_point.lon).cf.sel(T='2012-10').load()

In [ ]:
da1d.hvplot(x=t.name, grid=True)

Loading the entire time series at a point requires reading 669 chunks — we should use a Dask cluster.

### Parallelize with Dask
There are many ways to [deploy a Dask cluster](https://docs.dask.org/en/stable/deploying.html#deploy-dask-clusters). Choose one of the approaches below.

In [ ]:
#cluster_type = 'Local'
cluster_type = 'Coiled'
# cluster_type = 'Gateway'

#### Use LocalCluster

In [ ]:
if cluster_type == 'Local':
    from dask.distributed import LocalCluster, Client
    cluster = LocalCluster()
    client = Client(cluster)

#### Use Coiled

In [ ]:
if cluster_type == 'Coiled':
    import coiled
    cluster = coiled.Cluster(
        region='us-west-2',
        arm=True,
        worker_vm_types=['t4g.small'],
        worker_options={'nthreads': 2},
        n_workers=30,
        wait_for_workers=False,
        compute_purchase_option='spot_with_fallback',
        name='coawst-arm',
        software='coawst-icechunk-arm',
        workspace='esip-lab',
        timeout=180,
    )
    client = cluster.get_client()

#### Use Dask Gateway

In [ ]:
if cluster_type == 'Gateway':
    from dask_gateway import Gateway
    gateway = Gateway()
    options = gateway.cluster_options()
    options.conda_environment = 'global/global-pangeo'
    options.profile = 'Small Worker'
    cluster = gateway.new_cluster(options)
    client = cluster.get_client()
    cluster.adapt(minimum=4, maximum=30)

In [ ]:
client

Load the entire time series at a point:

In [ ]:
%%time
ds_selection = da.xoak.sel(lat_rho=ds_point.lat, lon_rho=ds_point.lon).load()

In [ ]:
ds_selection.hvplot(x=t.name, grid=True)

## Example: Compute the time mean for a variable over the entire domain for a specific time period

In [ ]:
%%time
da_mean = da.cf.sel(T=slice('2016-01-01 00:00', '2017-01-01 00:00')).mean(dim=t.name).compute()

In [ ]:
da_mean.hvplot.quadmesh(x=x.name, y=y.name, rasterize=True, geo=True, tiles='OSM', cmap='viridis')

## Example: Subset a time and space region and export to NetCDF

In [ ]:
def bbox2ij(lon, lat, bbox=[-160., -155., 18., 23.]):
    """Return indices for i,j that will completely cover the specified bounding box.
    i0,i1,j0,j1 = bbox2ij(lon,lat,bbox)
    """
    bbox = np.array(bbox)
    mypath = np.array([bbox[[0, 1, 1, 0]], bbox[[2, 2, 3, 3]]]).T
    p = path.Path(mypath)
    points = np.vstack((lon.ravel(), lat.ravel())).T
    n, m = np.shape(lon)
    inside = p.contains_points(points).reshape((n, m))
    ii, jj = np.meshgrid(range(m), range(n))
    return min(ii[inside]), max(ii[inside]), min(jj[inside]), max(jj[inside])

In [ ]:
bbox = [-76.63290610753754, -73.55671530588432, 37.57888442021855, 41.225532965406224]  # Delaware River Basin

In [ ]:
i0, i1, j0, j1 = bbox2ij(x.values, y.values, bbox=bbox)
print(i0, i1, j0, j1)

In [ ]:
ds_drb = ds[['temp', 'salt', 'Hwave']].isel(eta_rho=slice(j0, j1), xi_rho=slice(i0, i1))

In [ ]:
ds_drb

In [ ]:
ds_drb_timeslice = ds_drb.cf.sel(T=slice('2022-04-01 00:00', '2022-04-08 00:00'))

In [ ]:
ds_drb_timeslice = ds_drb_timeslice.chunk({'eta_rho': -1, 'xi_rho': -1})
print(f'Uncompressed dataset size: {ds_drb_timeslice.nbytes/1e6} MB')

In [ ]:
%%time
var = 'salt'
da_drb = ds_drb_timeslice[var].load()

In [ ]:
viz = da_drb.hvplot.quadmesh(x=x.name, y=y.name, geo=True,
                             cmap='turbo', rasterize=True, tiles='OSM', title=var)
viz = pn.panel(viz, widgets={'ocean_time': pn.widgets.Select})
pn.Column(viz).servable('DRB Explorer')

Close the Dask client before writing NetCDF (cannot write in parallel):

In [ ]:
client.close()

In [ ]:
%%time
encoding = {}
for var in ds_drb_timeslice.variables:
    encoding[var] = dict(zlib=True, complevel=4,
                         fletcher32=False, shuffle=True,
                         _FillValue=None)

ds_drb_timeslice.to_netcdf('drb.nc', encoding=encoding, mode='w')

## Stop cluster

In [ ]:
cluster.shutdown()